# Notebook 09: Leave-One-Subject-Out (LOSO) Cross-Validation

**Objective**: Measure cross-subject generalization across all four OPPORTUNITY subjects (S1, S2, S3, S4).

### Cross-Validation Protocol:
- 4 Folds: In Fold $i$, train strictly on 3 subjects and evaluate on held-out Subject $i$.
- **Strict Isolation**: Normalization statistics are fit strictly on the 3 training subjects for each fold to eliminate cross-subject data leakage.
- Evaluates Mean and Standard Deviation across folds.

## 1. Configure Experiment Mode

In [ ]:
# Set 'FAST' (2 runs per subject on CPU) or 'FULL' (all runs)
EXPERIMENT_MODE = "FAST"

import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

current = Path.cwd()
REPO_ROOT = current.parent if current.name == "notebooks" else current
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import src.api as api
api.setup_project()

## 2. Execute LOSO Cross-Validation (Random Forest Baseline on CPU)

In [ ]:
max_runs = 2 if EXPERIMENT_MODE == "FAST" else 5
loso_results = api.run_loso(model="random_forest", max_runs_per_sub=max_runs)

print("\n" + "=" * 75)
print(" LEAVE-ONE-SUBJECT-OUT (LOSO) CROSS-VALIDATION RESULTS")
print("=" * 75)
print(loso_results.to_string(index=False))
print("=" * 75)

## 3. Visualize Cross-Subject Generalization

In [ ]:
# Load persisted raw results for plotting
df_raw = pd.read_csv(REPO_ROOT / "results" / "loso" / "loso_random_forest_results.csv")
sub_df = df_raw[~df_raw["Held-Out Subject"].isin(["Mean", "Std Dev"])].copy()
mean_f1 = df_raw.loc[df_raw["Held-Out Subject"] == "Mean", "Macro F1"].values[0]

plt.figure(figsize=(7, 4))
sns.barplot(data=sub_df, x="Held-Out Subject", y="Macro F1", palette="Blues_d")
plt.axhline(mean_f1, color="red", linestyle="--", label=f"Mean Macro F1: {mean_f1*100:.2f}%")
plt.ylabel("Macro F1 Score")
plt.title("Leave-One-Subject-Out Cross-Validation Performance")
plt.legend()
plt.tight_layout()
plt.show()